# Format CuMiDa Datasets → re-annotated CuMiDa CSV

Re-annotates existing CuMiDa-format datasets (https://sbcb.inf.ufrgs.br/cumida)
using `src/dataset_builder.build_cumida_dataset` — the same treatment
`00.a_Format_GEO_Datasets` gives raw GEO series matrices, but starting from
an already-built probe-level CuMiDa CSV (`samples, type, probe_1, …`) instead
of a series matrix, since CuMiDa datasets ship pre-labelled (no series-matrix
parsing or class-characteristic discovery needed).

For each dataset it writes **5 files** into the dataset's raw folder:

| File | Contents |
|---|---|
| `<Tissue>_<GEO>_probe.csv` | **Dataset 2** — probe-level CuMiDa (AFFX-* controls removed) — feeds the benchmark |
| `<Tissue>_<GEO>_gene.csv`  | **Dataset 1** — gene-level CuMiDa (probes collapsed to gene symbols) |
| `<GEO>_metadata.csv`       | `sample_id, type` |
| `<GEO>_mapping.csv`        | `probID \| gene_symbol \| source \| dedup_status` — every original probe ID, including AFFX-* controls (marked `dedup_status="Removed_AFFX"`) |
| `<GEO>_annotation_report.txt` | coverage stats (annotation rows / mapped / unmapped / AFFX removed / unique probes & genes) |

Probe → gene mapping uses CuMiDa's own flat platform annotation files
(`Platform_Annotations/Cumida_platform/GPL*_limpo.txt.gz`), auto-detected
alongside GEO SOFT files by the same `parse_platform_annotation()`. Some
platforms (e.g. Agilent `GPL6848`/`GPL6480`) mix identifier namespaces across
features — a dataset's CSV header may use RefSeq/Ensembl accessions for some
probes and the platform's own design ID for others — so mapping tries `ID`,
`GB_ACC`, `REFSEQ`, and `ENSEMBL_ID` in turn (`extract_probe_gene_map_multi`).

**Config-driven:** build parameters for each CuMiDa origin live in
`configs/dataset_origin.yaml -> cumida_origins` — edit that file to add a new
dataset, then run this notebook. (Run-time dataset registration —
`enabled`, `platform`, `class_labels`, … — lives separately in
`configs/datasets.yaml`.)

## 1. Paths & imports

In [ ]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ directory.
REPO_ROOT = Path("..").resolve()                  # repo root (parent of notebooks/)
sys.path.insert(0, str(REPO_ROOT))                # make src.* importable

import pandas as pd
from src.helper import ConfigLoader, load_dataset
from src.dataset_builder import build_cumida_dataset

cfg = ConfigLoader(config_root=str(REPO_ROOT / "configs"))
print("Repo root:", REPO_ROOT)

## 2. Collect buildable datasets from config

Every entry in `configs/dataset_origin.yaml -> cumida_origins` is a
buildable CuMiDa CSV.

In [ ]:
cumida_origins_cfg = cfg.load_dataset_origin_config()["cumida_origins"]

build_specs = {}
for gse_id, oc in cumida_origins_cfg.items():
    origin = cfg.get_cumida_origin_config(gse_id)
    build_specs[gse_id] = {
        "gse_id": gse_id,
        "probe_csv": Path(origin["probe_csv"]),
        "annotation_file": Path(origin["annotation_file"]) if origin.get("annotation_file") else None,
        "platform": origin.get("platform"),
        "dedup_strategy": origin.get("dedup_strategy", "mean"),
        "use_mygene_fallback": origin.get("use_mygene_fallback", False),
        "species": origin.get("species", "human"),
        "tissue": origin["tissue"],
        "output_dir": Path(origin["output_dir"]),
    }

status_df = pd.DataFrame([
    {"gse": s["gse_id"], "tissue": s["tissue"], "platform": s["platform"],
     "probe_csv": s["probe_csv"].name, "exists": s["probe_csv"].exists()}
    for s in build_specs.values()
])
display(status_df)

## 3. Build all datasets

In [ ]:
statuses = []
for spec in build_specs.values():
    if not spec["probe_csv"].exists():
        print(f"SKIP {spec['gse_id']}: probe CSV not found at {spec['probe_csv']}.")
        continue
    status = build_cumida_dataset(
        probe_csv_path       = spec["probe_csv"],
        annotation_path      = spec["annotation_file"],
        output_dir           = spec["output_dir"],
        tissue               = spec["tissue"],
        geo_id               = spec["gse_id"],
        platform             = spec["platform"],
        dedup_strategy       = spec["dedup_strategy"],
        use_mygene_fallback  = spec["use_mygene_fallback"],
        species              = spec["species"],
    )
    statuses.append(status)

## 4. Summary

In [ ]:
summary = pd.DataFrame([
    {
        "GEO": s["geo_id"],
        "platform": s["platform"],
        "samples": s["samples_kept"],
        "probes_total": s["annotation_rows"],
        "probes_kept": s["unique_probes"],
        "genes": s["unique_genes"],
        "mapped": s["mapped_total"],
        "from_annot": s["mapped_from_annot"],
        "from_mygene": s["mapped_from_mygene"],
        "unmapped": s["unmapped"],
        "affx_removed": s["affx_removed"],
    }
    for s in statuses
])
display(summary)

for s in statuses:
    print(f"\n{s['geo_id']} class distribution: {s['class_distribution']}")

## 5. Sanity check — outputs load through the benchmark `data_loader`

In [ ]:
for s in statuses:
    probe_csv = s["output_files"]["probe_dataset"]
    gene_csv  = s["output_files"]["gene_dataset"]

    d_probe = load_dataset(probe_csv)
    d_gene  = load_dataset(gene_csv)

    print(f"=== {s['geo_id']} ===")
    print(f"  probe : X={d_probe['X'].shape}  classes={list(d_probe['label_encoder'].classes_)}")
    print(f"  gene  : X={d_gene['X'].shape}   first genes={d_gene['feature_names'][:5]}")
    print(f"  mapping: {s['output_files']['mapping']}")
    print(f"  report : {s['output_files']['report']}\n")